# Ring-trial data

> Load the ring-trial spectra and instruments, group the near-duplicate soils, and find the ring-trial soils that are in OSSL


In [ ]:
#| default_exp data

In [ ]:
#| export
import os
from pathlib import Path
import numpy as np, pandas as pd
from scipy.sparse.csgraph import connected_components
from soilspectfm.preprocessing import SNV
import ossl_rt_transfer
from soilspecdata.datasets.ringtrial import load_ringtrial

In [ ]:
from fastcore.test import *
from soilspecdata.datasets.ossl import load_ossl


The ring-trial data are not public. The cells that read them are marked `#| rtdata`, and `nbdev-test` skips them unless you run `nbdev-test --flags rtdata`. The outputs on this page come from a local run.

## Data location

In [ ]:
#| export
def data_path()->Path: # Folder holding the data shared by the soilspec projects
    "The `SOILSPEC_DATA` folder if that variable is set, else `data/` in the soilspec workspace"
    p = os.environ.get('SOILSPEC_DATA')
    return Path(p) if p else Path(ossl_rt_transfer.__file__).parents[2]/'data'

def rt_path()->Path: # Folder holding the ring-trial data
    "The `ringtrial` folder of `data_path()`"
    return data_path()/'ringtrial'


The soilspec workspace keeps the data in `data/`, beside the project folders. Set `SOILSPEC_DATA` to use a folder elsewhere. The ring-trial files are in its `ringtrial` folder, in the layout in which they were shared:

In [ ]:
#| rtdata
assert rt_path().is_dir()
sorted(o.name for o in rt_path().iterdir() if not o.name.startswith(('.', '~')))

['KSSL_soilMIRspectra_SNVplusSG1stDer.csv',
 'Spectrometers Metadata.xlsx',
 'drive-download-20231013T123706Z-001',
 'drive-download-20231013T123706Z-002']

## Spectra

soilspecdata's `load_ringtrial` returns one `Spectra` per instrument, keyed by the laboratory's `organization` name. Every `Spectra` has the same 70 soils, `RT_01` to `RT_70`, and the same wet chemistry as `source`. `Hokudai`, the Shimadzu instrument, has 59 of the soils:

In [ ]:
#| rtdata
rt = load_ringtrial(rt_path())
test_eq(len(rt), 21)
{k: v.spectra.shape for k, v in rt.items()}

{'Agrocares': (70, 1676),
 'Argonne': (70, 1676),
 'CSU_IL': (70, 1676),
 'ETH_alpha_1': (70, 1676),
 'ETH_alpha_2': (70, 1676),
 'ETH_vertex': (70, 1676),
 'IAEA_aug2022': (70, 1676),
 'KSSL': (70, 1676),
 'LandCare': (70, 1676),
 'Lesotho': (70, 1676),
 'MSU': (70, 1676),
 'OSU': (70, 1676),
 'Rothamsted': (70, 1676),
 'Scion': (70, 1676),
 'UGhent': (70, 1676),
 'UIUC': (70, 1676),
 'USP': (70, 1676),
 'UWisc_fine': (70, 1676),
 'Woodwell_alpha': (70, 1676),
 'Woodwell_vertex': (70, 1676),
 'Hokudai': (59, 1676)}

## Instruments

In [ ]:
#| export
_instrument_cols = {'unique_name': 'name', 'cluster': 'cluster', 'manufacturer': 'manufacturer', 'model': 'model',
                    'accessory': 'accessory', 'detector': 'detector', 'beamsplitter': 'beamsplitter',
                    'Spectral resolution (cm-1)': 'resolution', 'scans_per_spectra': 'scans',
                    'sample_presentation': 'presentation', 'country_iso': 'country'}

def load_instruments()->pd.DataFrame: # One row per ring-trial instrument, indexed by `organization`
    "Instrument metadata of the ring trial, with the spectral cluster of Safanelli et al. (2023)"
    df = pd.read_excel(rt_path()/'Spectrometers Metadata.xlsx', sheet_name='compiled MIR metadata')
    df = df.dropna(subset=['cluster']).set_index('folder_name').rename_axis('organization')
    return df[list(_instrument_cols)].rename(columns=_instrument_cols).astype({'cluster': int})

`load_instruments` reads the sheet `compiled MIR metadata` of `Spectrometers Metadata.xlsx`. It keeps the instrument columns and leaves out the contact details. Its index matches the keys of `load_ringtrial`. `cluster` is the group, 1 to 4, in which Safanelli et al. (2023) placed each instrument by the similarity of its spectra. `Hokudai` is not in the sheet, so it has no cluster:

In [ ]:
#| rtdata
inst = load_instruments()
test_eq(sorted(inst.index), sorted(k for k in rt if k != 'Hokudai'))
inst[['name', 'cluster', 'model', 'accessory', 'resolution']]

,name,cluster,model,accessory,resolution
organization,,,,,
LandCare,LandCare - Tensor II,4,Tensor II,Bruker HTS-XT,4.0
Woodwell_alpha,Woodwell - Alpha II,3,Alpha II,Bruker Alpha QuickSnap,4.0
ETH_vertex,ETHZ-SAE - Vertex 70,4,Vertex 70,Bruker HTS-XT,4.0
Argonne,Argonne - PerkinElmer Spectrum 100,2,Spectrum 100,Pike AutoDiff,4.0
UGhent,UGhent - Agilent 4300,2,4300 Handheld FTIR,Handheld FTIR,4.0
Agrocares,AgroCares - Alpha I,3,Alpha I,Bruker Alpha QuickSnap,8.0
ETH_alpha_1,ETHZ-SAE - Alpha I,3,Alpha I,Bruker Alpha QuickSnap,4.0
Woodwell_vertex,Woodwell - Vertex 70,3,Vertex 70,Pike AutoDiff,2.0
IAEA_aug2022,IAEA - Thermo Fisher Nicolet,2,Nicolet iS20,Collector II Diffuse Reflectance Accessory,4.0


The IAEA instrument, which also measured the Fukushima soils of `ossl-fuku-transfer`, is in cluster 2 with the PerkinElmer, Agilent and other Thermo Nicolet instruments. KSSL, the largest part of OSSL, is in cluster 4:

In [ ]:
#| rtdata
inst.groupby('cluster').size().to_dict(), inst.loc[['IAEA_aug2022', 'KSSL'], 'cluster'].to_dict()

({1: 1, 2: 6, 3: 6, 4: 7}, {'IAEA_aug2022': 2, 'KSSL': 4})

## Near-duplicate soils

In [ ]:
#| export
def near_dup_groups(
    spectra, # `Spectra` of one instrument
    min_corr=0.9999, # Smallest correlation of SNV spectra for two soils to join a group
)->np.ndarray: # Group label of each soil, in the order of `spectra.sample_ids`
    "Group soils whose SNV spectra correlate above `min_corr`, directly or through other soils of the group"
    X = SNV().fit_transform(spectra.spectra)
    corr = X @ X.T / X.shape[1]
    np.fill_diagonal(corr, 0)
    return connected_components(corr > min_corr, directed=False)[1]

Some ring-trial soils are near-duplicates: their spectra are almost the same, and so are their properties. A split must keep such soils on one side, or the test set holds soils that the model has seen in training. `near_dup_groups` links two soils when their SNV spectra correlate above `min_corr`, and gives each connected set of soils one label. On the KSSL instrument, three groups have more than one soil:

In [ ]:
#| rtdata
groups = near_dup_groups(rt['KSSL'])
ids = rt['KSSL'].sample_ids
dups = [list(ids[groups == g]) for g in np.unique(groups) if (groups == g).sum() > 1]
test_eq(dups, [['RT_04', 'RT_05', 'RT_07', 'RT_13', 'RT_15'], ['RT_10', 'RT_19'], ['RT_24', 'RT_25']])
dups

[['RT_04', 'RT_05', 'RT_07', 'RT_13', 'RT_15'],
 ['RT_10', 'RT_19'],
 ['RT_24', 'RT_25']]

The members of each group have similar properties, for example clay and pH:

In [ ]:
#| rtdata
rt['KSSL'].source.loc[[i for g in dups for i in g], ['clay_perc', 'pH_H20', 'potassium_cmolkg']]

,clay_perc,pH_H20,potassium_cmolkg
sample_id,,,
RT_04,17.10475,6.5300,0.49925
RT_05,17.22985,6.4700,0.59977
RT_07,17.07796,6.5900,0.48248
RT_13,16.95715,6.5300,0.56302
RT_15,15.98527,6.6000,0.61950
RT_10,21.88833,5.9025,0.37097
RT_19,21.09150,6.0300,0.40527
RT_24,12.71177,5.8500,0.18957
RT_25,12.99189,5.8800,0.18747


The threshold is close to the data: the most similar pair of soils outside these groups correlates at 0.99987. The groups come from the KSSL spectra, the reference instrument of the paper, and the same labels apply on every instrument.

## Ring-trial soils in OSSL

60 of the ring-trial soils come from KSSL, the largest part of OSSL. A model pretrained on all of OSSL has seen the ring-trial soils that OSSL contains, with their properties. Its scores on those soils are too good. `ossl_matches` finds, for each ring-trial soil, the OSSL sample whose SNV spectrum correlates best with it. It then compares their clay, pH and exchangeable K. A missing ring-trial value counts as the same.

In [ ]:
#| export
_ossl_props = {'clay_perc': 'clay.tot_usda.a334_w.pct', 'pH_H20': 'ph.h2o_usda.a268_index',
               'potassium_cmolkg': 'k.ext_usda.a725_cmolc.kg'} # Ring-trial property: the same property in OSSL

def ossl_matches(
    rt_spectra, # `Spectra` of one ring-trial instrument
    ossl_mir, # OSSL MIR `Spectra` on the same wavenumbers, such as `load_ossl().mir(650, 4000)`
    tol=1e-3, # Largest difference for two property values to count as the same
)->pd.DataFrame: # One row per ring-trial soil, indexed by sample ID
    "The OSSL sample whose SNV spectrum correlates best with each ring-trial soil, and whether their clay, pH and K are the same"
    assert np.array_equal(rt_spectra.wavenumbers, ossl_mir.wavenumbers), "The two `Spectra` need the same wavenumbers"
    snv = lambda X: (SNV().fit_transform(X) / np.sqrt(X.shape[1])).astype(np.float32)
    corr = snv(rt_spectra.spectra) @ snv(ossl_mir.spectra).T
    res = pd.DataFrame({'ossl_id': ossl_mir.sample_ids[corr.argmax(1)], 'corr': corr.max(1)},
                       index=pd.Index(rt_spectra.sample_ids, name='sample_id'))
    rt_props = rt_spectra.source.loc[res.index, list(_ossl_props)]
    ossl_props = ossl_mir.source.df.loc[res.ossl_id, list(_ossl_props.values())].to_numpy()
    same = np.isclose(rt_props.to_numpy(), ossl_props, atol=tol) | rt_props.isna().to_numpy()
    return res.assign(same_props=same.all(1))

On the KSSL instrument, 38 soils have an OSSL match with the same clay, pH and K. All 38 are KSSL soils. None of the 10 NAPT soils is in OSSL. `ossl_overlap` lists the 38 OSSL samples, which pretraining leaves out:

In [ ]:
#| export
ossl_overlap = [ # OSSL `id.layer_uuid_txt` of the 38 ring-trial soils that are in OSSL, to leave out of pretraining
    '166ad3fde03477823b58d211a5ae437b', '1b1bb1c411a882d92f64ae83db970171', '2260b64d4fc2de7cb67c6b87309ee0b6',
    '29526b18c6d81ee54d51afb72cc5cab4', '335be1b661037102856155fd3a8b1954', '378755c4961f3e0bba453dbf4807d4b8',
    '3c04cf70565ad2947f17a27fc0601637', '3d67736e6e87f44e9a9a87e16aa6a58d', '5438861f3fe9727750a36bb837af6bae',
    '559a6074eef9a6ad197e6b4e15ee0da5', '5e3b073f3da5f864d15411717155846e', '697f20d2fac2a09f015aeaefbddd88c8',
    '6a963ad214188cfadf1ac1d1724cfe33', '6db3c0b9e8ea92837c049062d33863fb', '6eccf77b5eef9991de782e84bec45150',
    '6fe5335e60a668e322e8c8475abc9b9e', '805a3fb21dfc4bc1579532168d0fa055', '925c79b0d0403a7c00542bc04739fda1',
    '935acf2551297075695a74781b6ff8e6', '94793b4066b3166846c0f0c2aea1355e', '987c0acfd5641043897ede4d61801f98',
    'a33bac6935b7c16839205678502be1e4', 'a3f04ddf41128963ab1f5d079406ab39', 'a97b39f4cac65313463e7d832db591ed',
    'ae9fdaa45114c6c5136fc1c1d2e6e2fc', 'af3ac37573b13fa79f0371c9b83e2421', 'b7904e56d0da59cb95f3f3ee10505418',
    'bbec9fff658d864d114f7802847b0e3e', 'c115baacb09bd3e7c1316ab4924fde11', 'c23f440eeed43ccda87dba08a867c041',
    'c674bc6243c2c67a84a865b05860ad3c', 'd15af0553652cd2761e76c3cabc76355', 'dfa38495a792115ef81e255c9da061fb',
    'e145f570a662d6060ef9b07437d9ec6f', 'ee72d2343f60c64c051f8f56f7c76261', 'f50d2740ee28c0b59fa073d75f543060',
    'f54594aab0b021afe53ed3afcbeaae6c', 'fd27fea3fff44267293129ba32550fde']

In [ ]:
#| rtdata
ossl_mir = load_ossl().mir(650, 4000)
matches = ossl_matches(rt['KSSL'], ossl_mir)
test_eq(sorted(matches.ossl_id[matches.same_props]), ossl_overlap)
matches.groupby(rt['KSSL'].source.loc[matches.index, 'source']).same_props.sum().to_dict()

{'KSSL': 38, 'NAPT': 0}

The 38 matches correlate at 0.99999 or more. The OSSL spectrum is a separate scan of the same soil, not a copy of the ring-trial spectrum. The best matches of the other soils correlate at up to 0.99992, for RT_10 and RT_23, but their properties differ. They stay in pretraining:

In [ ]:
#| rtdata
matches.groupby('same_props')['corr'].agg(['count', 'min', 'max'])

,count,min,max
same_props,,,
False,32,0.992424,0.999915
True,38,0.999991,0.999999
